In [ ]:
import pandas as pd

matches = pd.read_csv('data/matches.csv')
deliveries = pd.read_csv('data/deliveries.csv')

print(matches.columns.tolist())
print(deliveries.columns.tolist())

['id', 'Season', 'city', 'date', 'team1', 'team2', 'toss_winner', 'toss_decision', 'result', 'dl_applied', 'winner', 'win_by_runs', 'win_by_wickets', 'player_of_match', 'venue', 'umpire1', 'umpire2', 'umpire3']
['match_id', 'inning', 'batting_team', 'bowling_team', 'over', 'ball', 'batsman', 'non_striker', 'bowler', 'is_super_over', 'wide_runs', 'bye_runs', 'legbye_runs', 'noball_runs', 'penalty_runs', 'batsman_runs', 'extra_runs', 'total_runs', 'player_dismissed', 'dismissal_kind', 'fielder']


['id', 'Season', 'city', 'date', 'team1', 'team2', 'toss_winner', 'toss_decision', 'result', 'dl_applied', 'winner', 'win_by_runs', 'win_by_wickets', 'player_of_match', 'venue', 'umpire1', 'umpire2', 'umpire3']
['match_id', 'inning', 'batting_team', 'bowling_team', 'over', 'ball', 'batsman', 'non_striker', 'bowler', 'is_super_over', 'wide_runs', 'bye_runs', 'legbye_runs', 'noball_runs', 'penalty_runs', 'batsman_runs', 'extra_runs', 'total_runs', 'player_dismissed', 'dismissal_kind', 'fielder']


In [ ]:

match_info = matches[['id', 'winner', 'city']]


df = deliveries.merge(match_info, left_on='match_id', right_on='id')


df = df[df['inning'] == 2]


target = deliveries[deliveries['inning'] == 1].groupby('match_id')['total_runs'].sum().reset_index()
target.columns = ['match_id', 'target']

df = df.merge(target, on='match_id')

print("Data merged successfully!")
print(df.shape)

Data merged successfully!
(86240, 25)


In [ ]:

df['runs_so_far'] = df.groupby('match_id')['total_runs'].cumsum()


df['runs_left'] = df['target'] - df['runs_so_far']


df['balls_left'] = 120 - (df['over'] * 6 + df['ball'])


df['wickets_fallen'] = df.groupby('match_id')['player_dismissed'].cumcount()
df['wickets_left'] = 10 - df['wickets_fallen']


df['rrr'] = (df['runs_left'] / df['balls_left']) * 6


df['result'] = (df['batting_team'] == df['winner']).astype(int)

print("Features created!")
print(df[['batting_team', 'runs_left', 'balls_left', 'wickets_left', 'rrr', 'result']].head())

Features created!
                  batting_team  runs_left  balls_left  wickets_left  \
0  Royal Challengers Bangalore        206         113            10   
1  Royal Challengers Bangalore        206         112             9   
2  Royal Challengers Bangalore        206         111             8   
3  Royal Challengers Bangalore        204         110             7   
4  Royal Challengers Bangalore        200         109             6   

         rrr  result  
0  10.938053       0  
1  11.035714       0  
2  11.135135       0  
3  11.127273       0  
4  11.009174       0  


In [ ]:

df = df[df['balls_left'] > 0]
df = df[df['runs_left'] >= 0]
df = df.replace([float('inf'), float('-inf')], float('nan'))
df = df.dropna(subset=['runs_left', 'balls_left', 'wickets_left', 'rrr', 'result'])

print("Clean data shape:", df.shape)

Clean data shape: (83082, 32)


In [ ]:
from sklearn.model_selection import train_test_split

# Select only the columns we need
features = ['runs_left', 'balls_left', 'wickets_left', 'rrr']
target_col = 'result'

X = df[features]
y = df[target_col]

# Split into train and test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print("Training samples:", X_train.shape[0])
print("Testing samples:", X_test.shape[0])

Training samples: 66465
Testing samples: 16617


In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

model = RandomForestClassifier(n_estimators=50, random_state=42)
model.fit(X_train, y_train)

predictions = model.predict(X_test)
accuracy = accuracy_score(y_test, predictions)

print(f"Model Accuracy: {accuracy * 100:.2f}%")

Model Accuracy: 69.21%


In [14]:
import matplotlib.pyplot as plt
import pandas as pd

features = ['runs_left', 'balls_left', 'wickets_left', 'rrr']

importance = pd.Series(model.feature_importances_, index=features)

plt.figure(figsize=(6, 4))
importance.sort_values().plot(kind='barh', color='steelblue')
plt.title('Which factors matter most in predicting IPL win?')
plt.xlabel('Importance Score')
plt.tight_layout()
plt.show()

NotFittedError: This RandomForestClassifier instance is not fitted yet. Call 'fit' with appropriate arguments before using this estimator.